# 02 — Training Demo
Run the full training pipeline and observe live loss/accuracy updates.

In [1]:
import sys
sys.path.insert(0, '..')

from utils.config import load_config
from utils.helpers import set_seed, get_device
from data.dataset_loader import build_dataloaders
from data.preprocessing import get_train_transform, get_val_transform
from models.vl_jepa_model import VLJEPAModel
from training.trainer import Trainer
from utils.visualization import plot_training_history
import matplotlib.pyplot as plt
import os

print('Imports OK')

Imports OK


In [2]:
# ── Config ────────────────────────────────────────────────────────
cfg = load_config('../configs')

# Override for notebook (fewer epochs for quick demo)
cfg['training']['epochs']  = 10
cfg['training']['batch_size'] = 16

set_seed(cfg['project']['seed'])
device = get_device(cfg['device'])
print(f'Device: {device}')

Device: mps


In [3]:
data_raw = os.path.join('..', cfg['paths']['data_root'], 'train')

train_loader, val_loader, test_loader, vocab_size = build_dataloaders(
    data_root       = data_raw,   # 👈 ONLY TRAIN FOLDER
    train_transform = get_train_transform(cfg['dataset']['image_size']),
    val_transform   = get_val_transform(cfg['dataset']['image_size']),
    batch_size      = cfg['training']['batch_size'],
    num_workers     = 0,
    max_seq_len     = cfg['text_encoder']['max_seq_len'],
    seed            = cfg['project']['seed'],
    # 👇 DO NOT pass train_dir/test_dir
)

[dataset] Combining 1 source directory:
          • /Users/sreethanubhuvaneshgk/Downloads/desktop/home_folder/alzheimer's_detection/alzheimers_vl_jepa/data/raw/train
[dataset] Total images collected : 33984
          MildDemented             :   8960  (26.4%)
          ModerateDemented         :   6464  (19.0%)
          NonDemented              :   9600  (28.2%)
          VeryMildDemented         :   8960  (26.4%)

[dataset] Stratified split  →  Train:  23788  |  Val:   5097  |  Test:   5099



In [4]:
# ── Model ─────────────────────────────────────────────────────────
model = VLJEPAModel(
    vocab_size     = vocab_size,
    embedding_dim  = cfg['image_encoder']['embedding_dim'],
    projection_dim = cfg['vl_jepa']['projection_dim'],
    num_classes    = cfg['vl_jepa']['num_classes'],
    dropout        = cfg['vl_jepa']['dropout'],
    use_text       = cfg['vl_jepa']['use_text_branch'],
)

total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total:,}')

Trainable parameters: 639,716


In [5]:
print("Train sample:", train_loader.dataset.get_image_path(0))
print("Val sample:", val_loader.dataset.get_image_path(0))

Train sample: /Users/sreethanubhuvaneshgk/Downloads/desktop/home_folder/alzheimer's_detection/alzheimers_vl_jepa/data/raw/train/VeryMildDemented/357aa676-a62c-4229-959a-2a0642d512b8.jpg
Val sample: /Users/sreethanubhuvaneshgk/Downloads/desktop/home_folder/alzheimer's_detection/alzheimers_vl_jepa/data/raw/train/MildDemented/c914520e-9f3f-488b-98dc-e6adea59e490.jpg


In [6]:
train_paths = set([train_loader.dataset.get_image_path(i) for i in range(100)])
val_paths   = set([val_loader.dataset.get_image_path(i) for i in range(100)])

print("Overlap:", len(train_paths & val_paths))

Overlap: 0


In [7]:
# ── Train ─────────────────────────────────────────────────────────
trainer = Trainer(
    model            = model,
    train_loader     = train_loader,
    val_loader       = val_loader,
    device           = device,
    lr               = cfg['training']['learning_rate'],
    weight_decay     = cfg['training']['weight_decay'],
    epochs           = cfg['training']['epochs'],
    checkpoint_dir   = os.path.join('..', cfg['paths']['checkpoint_dir']),
    use_amp          = cfg['training']['use_amp'],
    early_stopping_patience = cfg['training']['early_stopping']['patience'],
)

history = trainer.fit()

Epoch [001/10]  Train Loss: 1.5286  Acc: 0.392 | Val Loss: 1.1233  Acc: 0.440  (181.2s)
  ✓ Checkpoint saved → ../experiments/checkpoints/best_model.pt


Epoch [002/10]  Train Loss: 1.3863  Acc: 0.482 | Val Loss: 0.9660  Acc: 0.561  (179.2s)
  ✓ Checkpoint saved → ../experiments/checkpoints/best_model.pt


Epoch [003/10]  Train Loss: 1.2532  Acc: 0.548 | Val Loss: 0.9430  Acc: 0.529  (180.8s)
  ✓ Checkpoint saved → ../experiments/checkpoints/best_model.pt


Epoch [004/10]  Train Loss: 1.1556  Acc: 0.590 | Val Loss: 0.8072  Acc: 0.634  (191.0s)
  ✓ Checkpoint saved → ../experiments/checkpoints/best_model.pt


Epoch [005/10]  Train Loss: 1.0987  Acc: 0.617 | Val Loss: 0.7784  Acc: 0.612  (181.6s)
  ✓ Checkpoint saved → ../experiments/checkpoints/best_model.pt


Epoch [006/10]  Train Loss: 1.0441  Acc: 0.640 | Val Loss: 0.7898  Acc: 0.634  (179.4s)


Epoch [007/10]  Train Loss: 1.0049  Acc: 0.659 | Val Loss: 0.6318  Acc: 0.713  (178.8s)
  ✓ Checkpoint saved → ../experiments/checkpoints/best_model.pt


Epoch [008/10]  Train Loss: 0.9657  Acc: 0.680 | Val Loss: 0.8002  Acc: 0.604  (178.4s)


Epoch [009/10]  Train Loss: 0.9382  Acc: 0.694 | Val Loss: 0.6834  Acc: 0.662  (179.2s)


Epoch [010/10]  Train Loss: 0.9196  Acc: 0.703 | Val Loss: 0.5809  Acc: 0.738  (180.1s)
  ✓ Checkpoint saved → ../experiments/checkpoints/best_model.pt

Best validation loss: 0.5809


In [9]:
# ── Plot training curves ──────────────────────────────────────────
epochs = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, history['train_loss'], label='Train', color='steelblue', linewidth=2)
ax1.plot(epochs, history['val_loss'],   label='Val',   color='tomato',    linewidth=2, linestyle='--')
ax1.set_title('Loss Curve'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, history['train_acc'], label='Train', color='seagreen', linewidth=2)
ax2.plot(epochs, history['val_acc'],   label='Val',   color='darkorange', linewidth=2, linestyle='--')
ax2.set_title('Accuracy Curve'); ax2.set_xlabel('Epoch'); ax2.set_ylim(0, 1); ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('VL-JEPA Training History', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nBest Val Acc  : {max(history["val_acc"]):.3f}')
print(f'Best Val Loss : {min(history["val_loss"]):.4f}')


Best Val Acc  : 0.738
Best Val Loss : 0.5809


/var/folders/td/71tf1gdd2_q3mzb3cs80h1lm0000gn/T/ipykernel_85720/3057016035.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
